# Lesson 10 - AI-agenter i produktion

I denne lektion vil du lære **produktionsmønstre** for AI-agenter ved hjælp af Microsoft Agent Framework (Python). Vi dækker:

- **Observabilitet** — tilføjelse af timing og logning til agentinteraktioner
- **Evaluering** — brug af en evaluatoragent til at bedømme responskvalitet
- **Omkostningsstyring** — strategier til tokenoptimering og modelvalg

Scenariet er en **rejseagent**, der hjælper brugere med at planlægge ture, med overvågning og evaluering lagdelt ovenpå.


## Opsætning


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Produktionsmæssige overvejelser

At flytte AI-agenter fra prototyper til produktion kræver nøje opmærksomhed på tre søjler:

1. **Observationsevne** — Du har brug for indsigt i, hvad agenten gør, hvor lang tid det tager, og hvilke værktøjer den kalder på. Uden sporing og logning er det næsten umuligt at fejlfinde problemer i produktion.

2. **Evaluering** — Automatiserede kvalitetskontroller sikrer, at agentens svar forbliver nøjagtige, komplette og hjælpsomme over tid. En evaluator-agent kan score svarene efter definerede kriterier.

3. **Omkostningsstyring** — Tokenforbrug påvirker direkte omkostningerne. Strategier som promptoptimering, modelvalg og caching hjælper med at holde udgifterne under kontrol uden at gå på kompromis med kvaliteten.


## Oprettelse af en Observerbar Agent

Vi definerer rejseværktøjer og pakker agentkaldet ind med timing, så vi kan overvåge latenstid. I produktion ville du integrere med OpenTelemetry eller en lignende sporings-backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evalueringsmønstre

Et almindeligt produktionsmønster er at bruge en anden agent som en **evaluator**. Evaluatoren bedømmer den primære agents svar ud fra foruddefinerede kriterier såsom fuldstændighed, nøjagtighed og hjælpsomhed.

Dette muliggør:
- Automatiserede kvalitetskontroller før svar når brugerne
- Regressionsovervågning når prompts eller modeller ændres
- Kontinuerlig overvågning af agentens ydeevne over tid


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Omkostningsstyringsstrategier

Kontrol med omkostninger er afgørende for produktions-AI-agenter. Her er nøglestrategier:

| Strategi | Beskrivelse |
|---|---|
| **Promptoptimering** | Hold systeminstruktioner korte. Fjern redundant kontekst for at reducere inputtokens. |
| **Modelvalg** | Brug mindre, billigere modeller (f.eks. GPT-4o-mini) til simple opgaver som klassificering eller udtræk, og reserver større modeller til kompleks ræsonnering. |
| **Caching** | Cache resultater fra værktøjer og hyppige forespørgsler for at undgå gentagne API-kald. |
| **Tokenbudgetter** | Sæt `max_tokens` grænser for at forhindre uventet lange svar. |
| **Batching** | Saml flere brugerforespørgsler i ét API-kald hvor det er muligt. |

I praksis fungerer en lagdelt tilgang godt: diriger enkle forespørgsler til en hurtig, billig model og eskaler kun komplekse forespørgsler til en mere kapabel model.


## Resumé

I denne lektion lærte du hvordan du:

1. **Tilføjer observabilitet** til agentinteraktioner med timing og logging, hvilket lægger grundlaget for tracing og overvågning.
2. **Evaluerer agenters svar** automatisk ved brug af en evaluator-agent, der vurderer fuldstændighed, nøjagtighed og hjælpsomhed.
3. **Styrer omkostninger** gennem promptoptimering, modelvalg, caching og tokenbudgetter.

Disse produktionsmønstre hjælper med at sikre, at dine AI-agenter er pålidelige, målbare og omkostningseffektive i stor skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
